In [1]:
from stereo.modeling.models.lightstereo.lightstereo import LightStereo

import torch
import torch_pruning as tp

from stereo.utils import common_utils
from easydict import EasyDict

import argparse


In [2]:
from argparse import Namespace

args = Namespace(
    dist_mode=False,
    cfg_file='cfgs/lightstereo/lightstereo_m_kitti.yaml',
    fix_random_seed=False,
    save_root_dir='./outputs',
    extra_tag='default',
    cover_old_exp=False,
    workers=0,
    pin_memory=False,
    run_mode='train',
    ckpt_dir = 'outputs/ckpts'
)

In [3]:
raw = common_utils.config_loader('cfgs/lightstereo/lightstereo_m_kitti.yaml')

raw = EasyDict(raw)


In [4]:
# Extract input dimensions from the loaded config for KITTI evaluation
h, w = 320, 736 # Default KITTI size from lightstereo_m_kitti.yaml
example_inputs = {
    'left': torch.randn(1, 3, h, w),
    'right': torch.randn(1, 3, h, w)
}
model = LightStereo(raw.MODEL)
forward_fn = lambda model, inputs: model(inputs)

DG = tp.DependencyGraph().build_dependency(
    model,
    example_inputs=example_inputs,
    forward_fn=forward_fn
)
print("Successfully built dependency graph.")

Successfully built dependency graph.


/home/samkit_jain/.conda/envs/rrc/lib/python3.8/site-packages/torch_pruning/dependency/shape_infer.py:52: UserWarning: Maximum recursive depth reached!
  warnings.warn("Maximum recursive depth reached!")


In [5]:
original_params = tp.utils.count_params(model)
print(f'Original model parameters: {original_params}')

Original model parameters: 7641264


In [6]:
# ============================================================================
# Load Pre-trained Checkpoint
# ============================================================================

checkpoint_path = 'LightStereo-M-KITTI.ckpt'

print(f"Loading pre-trained checkpoint: {checkpoint_path}")
print(f"{'='*70}")

try:
    # Load checkpoint
    checkpoint = torch.load(checkpoint_path, weights_only=True)
    
    # Handle different checkpoint formats
    if isinstance(checkpoint, dict):
        # Case 1: Checkpoint is a dict with 'state_dict' key
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
            print("  Found 'state_dict' in checkpoint")
        # Case 2: Checkpoint is a dict with model weights directly
        else:
            state_dict = checkpoint
            print("  Checkpoint contains model weights directly")
    else:
        # Case 3: Checkpoint is just the state dict
        state_dict = checkpoint
        print("  Checkpoint is state_dict")
    
    # Handle potential prefix mismatches (e.g., 'module.' prefix from DataParallel)
    # Remove 'module.' prefix if present
    cleaned_state_dict = {}
    for key, value in state_dict.items():
        new_key = key.replace('module.', '') if key.startswith('module.') else key
        cleaned_state_dict[new_key] = value
    
    # Load state dict into model
    missing_keys, unexpected_keys = model.load_state_dict(cleaned_state_dict, strict=False)
    
    print(f"\n  Checkpoint loaded successfully!")
    print(f"  Missing keys: {len(missing_keys)}")
    if missing_keys and len(missing_keys) <= 10:
        for key in missing_keys:
            print(f"    - {key}")
    
    print(f"  Unexpected keys: {len(unexpected_keys)}")
    if unexpected_keys and len(unexpected_keys) <= 10:
        for key in unexpected_keys:
            print(f"    - {key}")
    
    # Verify model loaded weights by checking a few parameters
    model_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n  Model parameters (trainable): {model_params:,}")
    
    # Test forward pass with loaded weights
    print(f"\n  Testing forward pass with pre-trained weights...")
    model.eval()
    with torch.no_grad():
        output = model(example_inputs)
    print(f"  ✓ Forward pass successful!")
    print(f"  Output keys: {list(output.keys())}")
    
    print(f"\n{'='*70}")
    print(f"Pre-trained model ready for pruning!")
    print(f"{'='*70}\n")
    
except FileNotFoundError:
    print(f"✗ Checkpoint file not found: {checkpoint_path}")
    print(f"  Please ensure the checkpoint file is in the current directory.")
    print(f"  Current directory: {os.getcwd()}")
except Exception as e:
    print(f"✗ Error loading checkpoint: {e}")
    import traceback
    traceback.print_exc()



Loading pre-trained checkpoint: LightStereo-M-KITTI.ckpt


/home/samkit_jain/.conda/envs/rrc/lib/python3.8/site-packages/torch/cuda/__init__.py:230: UserWarning: 
NVIDIA GeForce RTX 5060 Ti with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5060 Ti GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


  Checkpoint contains model weights directly

  Checkpoint loaded successfully!
  Missing keys: 802
  Unexpected keys: 1
    - model_state

  Model parameters (trainable): 7,641,264

  Testing forward pass with pre-trained weights...
  ✓ Forward pass successful!
  Output keys: ['disp_pred']

Pre-trained model ready for pruning!

  ✓ Forward pass successful!
  Output keys: ['disp_pred']

Pre-trained model ready for pruning!



In [7]:
from stereo.datasets import build_dataloader

print("Setting up KITTI dataloader...")
print(f"{'='*70}")

# Build the evaluation dataset from config
try:
    # Use the DATA_CONFIG from the loaded YAML
    data_config = raw.get('DATA_CONFIG', {})
    
    # Create dataset for evaluation/fine-tuning
    # Note: num_workers=0 to avoid multiprocessing issues in Jupyter notebook
    batch_size = 2  
    num_workers = 0  # Changed from 2 to 0 to avoid KittiDataset multiprocessing issues
    train_set, train_loader, train_sampler = build_dataloader(
        data_config,
        batch_size=batch_size,
        is_dist=False,
        workers=num_workers,
        pin_memory=False,
        mode = 'training')

    eval_set, eval_loader, eval_sampler = build_dataloader(
        data_config,
        batch_size=batch_size,
        is_dist=False,
        workers=num_workers,
        pin_memory=False,
        mode = 'evaluating')
    
    print(f"  ✓ KITTI dataloader created successfully!")
    print(f"  Dataset size: {len(train_set)}")
    print(f"  Batch size: {batch_size}")
    print(f"  Number of batches: {len(train_loader)}")
    print(f"  Num workers: {num_workers} (set to 0 to avoid multiprocessing issues)")
    
    use_dataloader = True
    
except Exception as e:
    print(f"  ✗ Could not create dataloader: {e}")
    print(f"  Will fall back to example inputs for fine-tuning")
    use_dataloader = False
    import traceback
    traceback.print_exc()

print(f"{'='*70}\n")

Setting up KITTI dataloader...
  ✓ KITTI dataloader created successfully!
  Dataset size: 194
  Batch size: 2
  Number of batches: 97
  Num workers: 0 (set to 0 to avoid multiprocessing issues)



In [8]:
for data in train_loader:
    print("Sample batch from KITTI training dataloader:")
    for key, value in data.items():
        if isinstance(value, torch.Tensor):
            print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
        else:
            print(f"  {key}: {value}")
    break  # Just show one batch

Sample batch from KITTI training dataloader:
  left: shape torch.Size([2, 3, 320, 736]), dtype torch.float32
  right: shape torch.Size([2, 3, 320, 736]), dtype torch.float32
  disp: shape torch.Size([2, 320, 736]), dtype torch.float32
  disp_right: shape torch.Size([2, 320, 736]), dtype torch.float32
  index: shape torch.Size([2]), dtype torch.int64
  name: ['/scratch/samkit_jain/kitti/data_stereo_flow/training/colored_0/000141_10.png', '/scratch/samkit_jain/kitti/data_stereo_flow/training/colored_0/000047_10.png']


In [9]:
# 2. Identify layers to ignore
# Critical layers for stereo matching that should NOT be pruned:
# Only protect the FINAL output layers (refine_3) and cost aggregation
# The backbone and other refinement layers can be pruned carefully

from stereo.modeling import build_trainer

ignored_layers = [
    model.refine_3,        # Final disparity refinement layer - MUST PROTECT
    model.cost_agg,        # Cost volume aggregation - MUST PROTECT
]

print("Layers to be ignored during pruning:")
for layer in ignored_layers:
    print(f"  - {layer.__class__.__name__}")
print(f"\nNote: Backbone, refine_1, and refine_2 will be pruned for better compression")

# 3. Configure iterative pruning parameters
total_pruning_ratio = 0.4  # Total target: prune 40% of weights
num_iterations = 5         # Number of iterative pruning steps
fine_tune_epochs = 5       # Fine-tune epochs per iteration (reduced to 1 for speed)

# Calculate per-iteration pruning ratio
# Each iteration prunes a fraction such that after all iterations, we reach total_pruning_ratio
per_iteration_ratio = 1 - (1 - total_pruning_ratio) ** (1 / num_iterations)

print(f"\nIterative Pruning Configuration:")
print(f"  - Total Target Pruning: {total_pruning_ratio * 100}%")
print(f"  - Number of Iterations: {num_iterations}")
print(f"  - Per-Iteration Ratio: {per_iteration_ratio * 100:.2f}%")
print(f"  - Fine-tune Epochs per Iteration: {fine_tune_epochs}")
print(f"  - Using real dataloader: {use_dataloader}")

# 5. Iterative pruning with fine-tuning loop
pruning_history = {'iteration': [], 'params_before': [], 'params_after': [], 'reduction': []}



Layers to be ignored during pruning:
  - BasicDeconv2d
  - Aggregation

Note: Backbone, refine_1, and refine_2 will be pruned for better compression

Iterative Pruning Configuration:
  - Total Target Pruning: 40.0%
  - Number of Iterations: 5
  - Per-Iteration Ratio: 9.71%
  - Fine-tune Epochs per Iteration: 5
  - Using real dataloader: True


In [11]:
import tqdm
import os
from torch.utils.tensorboard import SummaryWriter
import datetime



log_file = os.path.join('outputs', 'train_{}_{}.log'.format(datetime.datetime.now().strftime('%Y%m%d-%H%M%S'), 0))
logger = common_utils.create_logger(log_file, rank=0)
tb_writer = SummaryWriter(log_dir=os.path.join('outputs', 'tensorboard'))
common_utils.log_configs(raw, logger=logger)

model_trainer = build_trainer(args,raw, 0, 0, logger, tb_writer)
tbar = tqdm.trange(model_trainer.last_epoch + 1, model_trainer.total_epochs,
                       desc='epochs', dynamic_ncols=True,
                       bar_format='{l_bar}{bar}{r_bar}\n')

2025-10-25 21:01:22,706   INFO  ----------- DATA_CONFIG -----------
2025-10-25 21:01:22,706   INFO  cfgs.DATA_CONFIG.DATA_INFOS: [{'DATASET': 'KittiDataset', 'DATA_SPLIT': {'TRAINING': './data/KITTI12/kitti12_train194.txt', 'EVALUATING': './data/KITTI12/kitti12_val14.txt', 'TESTING': './data/KITTI12/kitti12_test.txt'}, 'RETURN_RIGHT_DISP': True}]
2025-10-25 21:01:22,707   INFO  ----------- DATA_TRANSFORM -----------
2025-10-25 21:01:22,707   INFO  cfgs.DATA_CONFIG.DATA_TRANSFORM.TRAINING: [{'NAME': 'StereoColorJitter', 'BRIGHTNESS': [0.7, 1.3], 'CONTRAST': [0.7, 1.3], 'SATURATION': [0.7, 1.3], 'HUE': [-0.3, 0.3], 'ASYMMETRIC_PROB': 0}, {'NAME': 'RandomErase', 'PROB': 0.5, 'MAX_TIME': 2, 'BOUNDS': [50, 100]}, {'NAME': 'RandomSparseScale', 'SIZE': [320, 736], 'MIN_SCALE': 0.2, 'MAX_SCALE': 0.5, 'SCALE_PROB': 0.8}, {'NAME': 'RandomCrop', 'SIZE': [320, 736]}, {'NAME': 'TransposeImage'}, {'NAME': 'ToTensor'}, {'NAME': 'NormalizeImage', 'MEAN': [0.485, 0.456, 0.406], 'STD': [0.229, 0.224, 0.

In [12]:
print(f"\n{'='*70}")
print(f"Starting Iterative Pruning and Fine-tuning")
print(f"{'='*70}\n")

for iteration in range(num_iterations):
    print(f"{'─'*70}")
    print(f"ITERATION {iteration + 1}/{num_iterations}")
    print(f"{'─'*70}")
    
    # Record parameters before pruning
    params_before = tp.utils.count_params(model)
    
    # 5a. Create pruner for this iteration
    pruner = tp.pruner.MagnitudePruner(
        model,
        example_inputs=example_inputs,
        importance=tp.importance.MagnitudeImportance(),
        iterative_steps=1,  # Single step per iteration
        pruning_ratio=per_iteration_ratio,
        ignored_layers=ignored_layers,
        forward_fn=forward_fn,
    )
    
    # 5b. Execute pruning
    print(f"  [Step 1/3] Pruning with ratio: {per_iteration_ratio * 100:.2f}%")
    pruner.step()
    
    params_after = tp.utils.count_params(model)
    iteration_reduction = (params_before - params_after) / params_before * 100
    print(f"    Parameters: {params_before:,} → {params_after:,} ({iteration_reduction:.2f}% removed)")
    
    # 🔁 Rebuild optimizer, scheduler, and warmup after pruning
    model_trainer.optimizer, model_trainer.scheduler = model_trainer.build_optimizer_and_scheduler()
    model_trainer.warmup_scheduler = model_trainer.build_warmup()
    
    # 5c. Fine-tuning with real data
    print(f"  [Step 2/3] Fine-tuning for {fine_tune_epochs} epochs...")
    model.train()
    
    for current_epoch in tbar:
        model_trainer.train(current_epoch, tbar)
        model_trainer.save_ckpt(current_epoch)
        if current_epoch % raw.TRAINER.EVAL_INTERVAL == 0 or current_epoch == model_trainer.total_epochs - 1:
            model_trainer.evaluate(current_epoch)
    
    # 5d. Verify model after fine-tuning
    print(f"  [Step 3/3] Verifying pruned model...")
    model.eval()
    try:
        with torch.no_grad():
            output = model(example_inputs)
        print(f"    ✓ Forward pass successful!")
        if 'disp_pred' in output:
            print(f"    ✓ Output shape: {output['disp_pred'].shape}")
    except Exception as e:
        print(f"    ✗ Error: {e}")
    
    # Record metrics
    pruning_history['iteration'].append(iteration + 1)
    pruning_history['params_before'].append(params_before)
    pruning_history['params_after'].append(params_after)
    pruning_history['reduction'].append(iteration_reduction)
    
    print()

# 6. Summary of pruning process
print(f"{'='*70}")
print(f"Iterative Pruning Complete - Summary")
print(f"{'='*70}")
print(f"{'Iter':<6} {'Before':<15} {'After':<15} {'Removed':<15}")
print(f"{'─'*70}")

for i, (it, pb, pa, red) in enumerate(zip(
    pruning_history['iteration'],
    pruning_history['params_before'],
    pruning_history['params_after'],
    pruning_history['reduction']
)):
    print(f"{it:<6} {pb:<15,} {pa:<15,} {red:<14.2f}%")

# 7. Final analysis
final_params = tp.utils.count_params(model)
total_reduction = (original_params - final_params) / original_params * 100

print(f"\n{'='*70}")
print(f"Final Pruning Results:")
print(f"{'='*70}")
print(f"  Original Parameters:     {original_params:,}")
print(f"  Final Parameters:        {final_params:,}")
print(f"  Total Parameters Removed: {original_params - final_params:,}")
print(f"  Total Reduction Rate:    {total_reduction:.2f}%")
print(f"{'='*70}")

# 8. Final verification
print(f"\nPerforming final model verification...")
model.eval()
try:
    with torch.no_grad():
        output = model(example_inputs)
    print("✓ Final forward pass successful!")
    print(f"  Output keys: {list(output.keys())}")
    if 'disp_pred' in output:
        print(f"  Disparity prediction shape: {output['disp_pred'].shape}")
except Exception as e:
    print(f"✗ Error during final verification: {e}")
    import traceback
    traceback.print_exc()



Starting Iterative Pruning and Fine-tuning

──────────────────────────────────────────────────────────────────────
ITERATION 1/5
──────────────────────────────────────────────────────────────────────
  [Step 1/3] Pruning with ratio: 9.71%
  [Step 1/3] Pruning with ratio: 9.71%
    Parameters: 7,641,264 → 7,284,229 (4.67% removed)
  [Step 2/3] Fine-tuning for 5 epochs...
    Parameters: 7,641,264 → 7,284,229 (4.67% removed)
  [Step 2/3] Fine-tuning for 5 epochs...


/home/samkit_jain/ml/OpenStereoSLAM/stereo/modeling/trainer_template.py:210: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.cfgs.OPTIMIZATION.AMP):
2025-10-25 21:03:25,371   INFO  Training Epoch: 0/100 Iter:   0/97 Loss:0.956940(0.956940) LR:8.0000e-06 DataTime:0.22 InferTime:86475.04ms Time cost: 02:02/329:49:25
2025-10-25 21:03:25,371   INFO  Training Epoch: 0/100 Iter:   0/97 Loss:0.956940(0.956940) LR:8.0000e-06 DataTime:0.22 InferTime:86475.04ms Time cost: 02:02/329:49:25
2025-10-25 21:03:42,075   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.275387(0.668727) LR:1.1028e-04 DataTime:0.20 InferTime:27.11ms Time cost: 02:19/7:18:41
2025-10-25 21:03:42,075   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.275387(0.668727) LR:1.1028e-04 DataTime:0.20 InferTime:27.11ms Time cost: 02:19/7:18:41
/home/samkit_jain/ml/OpenStereoSLAM/stereo/modeling/trainer_template.py:28

  [Step 3/3] Verifying pruned model...
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 2/5
──────────────────────────────────────────────────────────────────────
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 2/5
──────────────────────────────────────────────────────────────────────
  [Step 1/3] Pruning with ratio: 9.71%
  [Step 1/3] Pruning with ratio: 9.71%
    Parameters: 7,284,229 → 6,998,028 (3.93% removed)
  [Step 2/3] Fine-tuning for 5 epochs...
    Parameters: 7,284,229 → 6,998,028 (3.93% removed)
  [Step 2/3] Fine-tuning for 5 epochs...


2025-10-25 21:59:02,503   INFO  Training Epoch: 0/100 Iter:   0/97 Loss:0.179270(0.179270) LR:8.0000e-06 DataTime:0.23 InferTime:27.31ms Time cost: 57:39/9320:36:41
2025-10-25 21:59:19,559   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.292775(0.376761) LR:1.1028e-04 DataTime:0.22 InferTime:27.09ms Time cost: 57:56/182:42:40
2025-10-25 21:59:19,559   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.292775(0.376761) LR:1.1028e-04 DataTime:0.22 InferTime:27.09ms Time cost: 57:56/182:42:40
2025-10-25 21:59:35,506   INFO  Evaluating Epoch: 0 Iter:   0/14 InferTime: 17.54ms
2025-10-25 21:59:35,506   INFO  Evaluating Epoch: 0 Iter:   0/14 InferTime: 17.54ms
2025-10-25 21:59:36,729   INFO  Epoch 0 metrics: {'d1_all': tensor(1.7140), 'epe': tensor(0.5947), 'thres_1': tensor(11.6733), 'thres_2': tensor(3.8444), 'thres_3': tensor(2.0426)}
2025-10-25 21:59:36,729   INFO  Epoch 0 metrics: {'d1_all': tensor(1.7140), 'epe': tensor(0.5947), 'thres_1': tensor(11.6733), 'thres_2': tensor(3.8444), 'thr

  [Step 3/3] Verifying pruned model...
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 3/5
──────────────────────────────────────────────────────────────────────
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 3/5
──────────────────────────────────────────────────────────────────────
  [Step 1/3] Pruning with ratio: 9.71%
  [Step 1/3] Pruning with ratio: 9.71%
    Parameters: 6,998,028 → 6,765,707 (3.32% removed)
  [Step 2/3] Fine-tuning for 5 epochs...
    Parameters: 6,998,028 → 6,765,707 (3.32% removed)
  [Step 2/3] Fine-tuning for 5 epochs...


...

  [Step 3/3] Verifying pruned model...
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 4/5
──────────────────────────────────────────────────────────────────────
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 4/5
──────────────────────────────────────────────────────────────────────
  [Step 1/3] Pruning with ratio: 9.71%
  [Step 1/3] Pruning with ratio: 9.71%
    Parameters: 6,765,707 → 6,575,208 (2.82% removed)
  [Step 2/3] Fine-tuning for 5 epochs...
    Parameters: 6,765,707 → 6,575,208 (2.82% removed)
  [Step 2/3] Fine-tuning for 5 epochs...


2025-10-25 23:50:07,308   INFO  Training Epoch: 0/100 Iter:   0/97 Loss:0.199534(0.199534) LR:8.0000e-06 DataTime:0.23 InferTime:27.37ms Time cost: 2:48:44/27276:42:32
2025-10-25 23:50:24,399   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.210678(0.317920) LR:1.1028e-04 DataTime:0.19 InferTime:27.22ms Time cost: 2:49:01/532:58:42
2025-10-25 23:50:24,399   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.210678(0.317920) LR:1.1028e-04 DataTime:0.19 InferTime:27.22ms Time cost: 2:49:01/532:58:42
2025-10-25 23:50:40,469   INFO  Evaluating Epoch: 0 Iter:   0/14 InferTime: 18.09ms
2025-10-25 23:50:40,469   INFO  Evaluating Epoch: 0 Iter:   0/14 InferTime: 18.09ms
2025-10-25 23:50:41,749   INFO  Epoch 0 metrics: {'d1_all': tensor(1.5729), 'epe': tensor(0.5476), 'thres_1': tensor(10.9680), 'thres_2': tensor(3.7019), 'thres_3': tensor(1.9603)}
2025-10-25 23:50:41,749   INFO  Epoch 0 metrics: {'d1_all': tensor(1.5729), 'epe': tensor(0.5476), 'thres_1': tensor(10.9680), 'thres_2': tensor(3.7019

  [Step 3/3] Verifying pruned model...
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 5/5
──────────────────────────────────────────────────────────────────────
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

──────────────────────────────────────────────────────────────────────
ITERATION 5/5
──────────────────────────────────────────────────────────────────────
  [Step 1/3] Pruning with ratio: 9.71%
  [Step 1/3] Pruning with ratio: 9.71%
    Parameters: 6,575,208 → 6,417,848 (2.39% removed)
  [Step 2/3] Fine-tuning for 5 epochs...
    Parameters: 6,575,208 → 6,417,848 (2.39% removed)
  [Step 2/3] Fine-tuning for 5 epochs...


2025-10-26 00:45:47,729   INFO  Training Epoch: 0/100 Iter:   0/97 Loss:0.234430(0.234430) LR:8.0000e-06 DataTime:0.22 InferTime:27.51ms Time cost: 3:44:24/36276:21:29
2025-10-26 00:46:04,911   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.184070(0.299443) LR:1.1028e-04 DataTime:0.22 InferTime:27.27ms Time cost: 3:44:41/708:32:14
2025-10-26 00:46:04,911   INFO  Training Epoch: 0/100 Iter:  50/97 Loss:0.184070(0.299443) LR:1.1028e-04 DataTime:0.22 InferTime:27.27ms Time cost: 3:44:41/708:32:14
2025-10-26 00:46:21,054   INFO  Evaluating Epoch: 0 Iter:   0/14 InferTime: 17.72ms
2025-10-26 00:46:21,054   INFO  Evaluating Epoch: 0 Iter:   0/14 InferTime: 17.72ms
2025-10-26 00:46:22,301   INFO  Epoch 0 metrics: {'d1_all': tensor(1.5262), 'epe': tensor(0.5503), 'thres_1': tensor(11.1515), 'thres_2': tensor(3.8228), 'thres_3': tensor(2.0053)}
2025-10-26 00:46:22,301   INFO  Epoch 0 metrics: {'d1_all': tensor(1.5262), 'epe': tensor(0.5503), 'thres_1': tensor(11.1515), 'thres_2': tensor(3.8228

  [Step 3/3] Verifying pruned model...
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

Iterative Pruning Complete - Summary
Iter   Before          After           Removed        
──────────────────────────────────────────────────────────────────────
1      7,641,264       7,284,229       4.67          %
2      7,284,229       6,998,028       3.93          %
3      6,998,028       6,765,707       3.32          %
4      6,765,707       6,575,208       2.82          %
5      6,575,208       6,417,848       2.39          %

Final Pruning Results:
  Original Parameters:     7,641,264
  Final Parameters:        6,417,848
  Total Parameters Removed: 1,223,416
  Total Reduction Rate:    16.01%

Performing final model verification...
✓ Final forward pass successful!
  Output keys: ['disp_pred']
  Disparity prediction shape: torch.Size([1, 1, 320, 736])
    ✓ Forward pass successful!
    ✓ Output shape: torch.Size([1, 1, 320, 736])

Iterative Pruning Complete - S

In [13]:
model_trainer.evaluate(model_trainer.last_epoch)
pruned_params = tp.utils.count_params(model)
print(f'Final pruned model parameters: {pruned_params}')

2025-10-26 01:42:46,723   INFO  Evaluating Epoch:-1 Iter:   0/14 InferTime: 17.69ms
2025-10-26 01:42:47,959   INFO  Epoch -1 metrics: {'d1_all': tensor(1.3987), 'epe': tensor(0.5054), 'thres_1': tensor(9.7965), 'thres_2': tensor(3.2704), 'thres_3': tensor(1.7089)}
2025-10-26 01:42:47,959   INFO  Epoch -1 metrics: {'d1_all': tensor(1.3987), 'epe': tensor(0.5054), 'thres_1': tensor(9.7965), 'thres_2': tensor(3.2704), 'thres_3': tensor(1.7089)}


Final pruned model parameters: 6417848


In [14]:
original_model_trainer = build_trainer(args, raw, 0, 0, logger, tb_writer)

original_model_trainer.evaluate(original_model_trainer.last_epoch)

2025-10-26 01:42:52,694   INFO  Loading parameters from checkpoint LightStereo-M-KITTI.ckpt
2025-10-26 01:42:52,796   INFO  Unused weight: 
2025-10-26 01:42:52,797   INFO  Not updated weight: 
2025-10-26 01:42:52,799   INFO  Total samples for eval dataset: 14
2025-10-26 01:42:52,800   INFO  Total samples for train dataset: 194
2025-10-26 01:42:52,796   INFO  Unused weight: 
2025-10-26 01:42:52,797   INFO  Not updated weight: 
2025-10-26 01:42:52,799   INFO  Total samples for eval dataset: 14
2025-10-26 01:42:52,800   INFO  Total samples for train dataset: 194
2025-10-26 01:42:52,875   INFO  Evaluating Epoch:-1 Iter:   0/14 InferTime: 21.07ms
2025-10-26 01:42:52,875   INFO  Evaluating Epoch:-1 Iter:   0/14 InferTime: 21.07ms
2025-10-26 01:42:54,109   INFO  Epoch -1 metrics: {'d1_all': tensor(1.5587), 'epe': tensor(0.5167), 'thres_1': tensor(9.9214), 'thres_2': tensor(3.3797), 'thres_3': tensor(1.8467)}
2025-10-26 01:42:54,109   INFO  Epoch -1 metrics: {'d1_all': tensor(1.5587), 'epe': t

In [15]:
import pandas as pd
import numpy as np

print("="*80)
print("COMPREHENSIVE PRUNING RESULTS COMPARISON")
print("="*80)

# ============================================================================
# 1. OVERALL COMPRESSION SUMMARY
# ============================================================================
print("\n1. OVERALL COMPRESSION SUMMARY")
print("─"*80)

total_params_removed = original_params - pruned_params
total_reduction_rate = (total_params_removed / original_params) * 100
compression_ratio = original_params / pruned_params if pruned_params > 0 else 0

print(f"  Original Model Parameters:        {original_params:>15,}")
print(f"  Final Pruned Model Parameters:    {pruned_params:>15,}")
print(f"  Total Parameters Removed:         {total_params_removed:>15,}")
print(f"  Total Reduction Rate:             {total_reduction_rate:>14.2f}%")
print(f"  Compression Ratio (Original/Final): {compression_ratio:>11.2f}x")
print(f"  Memory Savings:                   {(1 - pruned_params/original_params)*100:>14.2f}%")

# ============================================================================
# 2. ITERATIVE PRUNING BREAKDOWN (by iteration)
# ============================================================================
print("\n2. ITERATIVE PRUNING BREAKDOWN")
print("─"*80)

# Create detailed table
if pruning_history and 'iteration' in pruning_history and len(pruning_history['iteration']) > 0:
    df_history = pd.DataFrame({
        'Iteration': pruning_history['iteration'],
        'Params Before': pruning_history['params_before'],
        'Params After': pruning_history['params_after'],
        'Removed This Iter': [b - a for b, a in zip(pruning_history['params_before'], 
                                                       pruning_history['params_after'])],
        'Reduction %': pruning_history['reduction'],
    })
    
    # Add cumulative reduction
    cumulative_reduction = [(original_params - pa) / original_params * 100 
                           for pa in df_history['Params After']]
    df_history['Cumulative Reduction %'] = cumulative_reduction
    
    # Display with formatting
    print(f"{'Iter':<6} {'Before':<15} {'After':<15} {'Removed':<15} {'Per-Iter %':<12} {'Cumul %':<12}")
    print("─"*80)
    
    for idx, row in df_history.iterrows():
        print(f"{row['Iteration']:<6} {row['Params Before']:<15,} {row['Params After']:<15,} "
              f"{row['Removed This Iter']:<15,} {row['Reduction %']:<11.2f}% {row['Cumulative Reduction %']:<11.2f}%")
    
    print("\n  Summary Statistics:")
    print(f"    Average parameters removed per iteration: {df_history['Removed This Iter'].mean():,.0f}")
    print(f"    Min reduction in single iteration:       {df_history['Reduction %'].min():.2f}%")
    print(f"    Max reduction in single iteration:       {df_history['Reduction %'].max():.2f}%")
    print(f"    Avg reduction per iteration:             {df_history['Reduction %'].mean():.2f}%")
    
else:
    print("  ⚠ No pruning history available (Cell 11 may not have been executed)")

# ============================================================================
# 3. TARGET VS ACTUAL RESULTS
# ============================================================================
print("\n3. TARGET VS ACTUAL PRUNING RESULTS")
print("─"*80)

print(f"  Target Compression:     {total_pruning_ratio * 100:>14.2f}%")
print(f"  Actual Compression:     {total_reduction_rate:>14.2f}%")
print(f"  Difference:             {(total_reduction_rate - total_pruning_ratio*100):>+14.2f}%")

if total_reduction_rate >= total_pruning_ratio * 100 * 0.95:
    status = "✓ EXCELLENT - Close to target"
elif total_reduction_rate >= total_pruning_ratio * 100 * 0.90:
    status = "✓ GOOD - Within acceptable range"
else:
    status = "⚠ WARNING - Below target compression"

print(f"  Status:                 {status}")

# ============================================================================
# 4. MODEL SIZE REDUCTION (in MB, assuming float32)
# ============================================================================
print("\n4. MODEL SIZE REDUCTION (Estimated)")
print("─"*80)

# Assume float32 (4 bytes per parameter)
bytes_per_param = 4  # float32

original_size_mb = (original_params * bytes_per_param) / (1024 * 1024)
pruned_size_mb = (pruned_params * bytes_per_param) / (1024 * 1024)
size_saved_mb = original_size_mb - pruned_size_mb

print(f"  Original Model Size:    {original_size_mb:>15.2f} MB")
print(f"  Pruned Model Size:      {pruned_size_mb:>15.2f} MB")
print(f"  Size Saved:             {size_saved_mb:>15.2f} MB")
print(f"  Size Reduction:         {(size_saved_mb/original_size_mb)*100:>14.2f}%")

# ============================================================================
# 5. EFFICIENCY METRICS
# ============================================================================
print("\n5. PRUNING EFFICIENCY METRICS")
print("─"*80)

print(f"  Compression Ratio:      {compression_ratio:>14.2f}x")
print(f"  Inference Speedup:      ~{compression_ratio:>13.2f}x (theoretical, no sparsity)")
print(f"  Memory Bandwidth Savings: {(1-1/compression_ratio)*100:>10.2f}%")

# ============================================================================
# 6. LAYER-BY-LAYER ANALYSIS (from pruning history)
# ============================================================================
print("\n6. PRUNING PROGRESS PER ITERATION")
print("─"*80)

if pruning_history and 'iteration' in pruning_history and len(pruning_history['iteration']) > 0:
    print(f"  Total Iterations Completed: {len(pruning_history['iteration'])}")
    print(f"  Target Iterations:          {num_iterations}")
    
    if len(pruning_history['iteration']) == num_iterations:
        print(f"  Status: ✓ All iterations completed")
    else:
        print(f"  Status: ⚠ Only {len(pruning_history['iteration'])}/{num_iterations} iterations completed")
    
    print("\n  Iteration-by-iteration breakdown:")
    for i, (it, pb, pa, red) in enumerate(zip(
        pruning_history['iteration'],
        pruning_history['params_before'],
        pruning_history['params_after'],
        pruning_history['reduction']
    )):
        # Calculate parameters removed in this specific iteration
        params_removed = pb - pa
        
        # Calculate what percentage of remaining params were pruned
        percent_of_remaining = (params_removed / pb) * 100
        
        print(f"    Iter {it}: {params_removed:>9,} params removed ({percent_of_remaining:>6.2f}% of iteration start)")

# ============================================================================
# 7. VISUALIZATION TABLE (ASCII)
# ============================================================================
print("\n7. FINAL SUMMARY TABLE")
print("─"*80)
print(f"{'Metric':<40} {'Original':<20} {'Pruned':<20}")
print("─"*80)
print(f"{'Total Parameters':<40} {original_params:>18,} {pruned_params:>18,}")
print(f"{'Model Size (MB, float32)':<40} {original_size_mb:>18.2f} {pruned_size_mb:>18.2f}")
print(f"{'Parameters Per MB':<40} {original_params/original_size_mb:>18,.0f} {pruned_params/pruned_size_mb:>18,.0f}")
print(f"{'Compression Ratio':<40} {'1.0x':>18} {f'{compression_ratio:.2f}x':>18}")
print(f"{'Reduction Rate':<40} {'0.00%':>18} {f'{total_reduction_rate:.2f}%':>18}")

print("\n" + "="*80)
print("PRUNING ANALYSIS COMPLETE")
print("="*80)


COMPREHENSIVE PRUNING RESULTS COMPARISON

1. OVERALL COMPRESSION SUMMARY
────────────────────────────────────────────────────────────────────────────────
  Original Model Parameters:              7,641,264
  Final Pruned Model Parameters:          6,417,848
  Total Parameters Removed:               1,223,416
  Total Reduction Rate:                      16.01%
  Compression Ratio (Original/Final):        1.19x
  Memory Savings:                            16.01%

2. ITERATIVE PRUNING BREAKDOWN
────────────────────────────────────────────────────────────────────────────────
Iter   Before          After           Removed         Per-Iter %   Cumul %     
────────────────────────────────────────────────────────────────────────────────
1.0    7,641,264.0     7,284,229.0     357,035.0       4.67       % 4.67       %
2.0    7,284,229.0     6,998,028.0     286,201.0       3.93       % 8.42       %
3.0    6,998,028.0     6,765,707.0     232,321.0       3.32       % 11.46      %
4.0    6,765,707.